Configuracion del entorno

In [ ]:

!pip install -q torch transformers datasets tokenizers evaluate accelerate


import torch
import transformers
import datasets
import random 


if torch.cuda.is_available():
    # Crea un objeto 'device' que apunte a la GPU principal
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    print(f" Éxito: GPU detectada y configurada: {gpu_name}")
    print(f" Versión de CUDA disponible para PyTorch: {torch.version.cuda}")
else:

    print(" No se detectó una GPU ")
    device = torch.device("none")



# Limpia la caché de la GPU 
# util en recompilacion, comentar si es primera vez

if device.type == 'cuda': 
#chocotavo dame tu gpu 
    torch.cuda.empty_cache()
    print(" Memoria de la GPU liberada y lista para trabajar.")


Utilizamos CUDA, es lo mismo que ya se vio en clase
torch: Es la misma biblioteca que vimos  
transformers: Es la biblioteca de Hugging Face, tiene varias herramientas interesantes apra NLP de ahi vienen todas las siguientes herramientas 
datasets:para manejar conjuntos de datos
tokenizers:  para tokenizar texto 
evaluate: para evaluar modelos 
accelerate: para acelerar el entrenamiento de modelos mediante técnicas como mixed precision y distribución en múltiples GPUs

In [ ]:
import os
from datasets import Dataset


ruta_es = "./IALATIN/Raw_Data/CCMatrix.es-la.es"
ruta_la = "./IALATIN/Raw_Data/CCMatrix.es-la.la"
ruta_scores = "./IALATIN/Raw_Data/CCMatrix.es-la.scores"




if all(os.path.exists(ruta) for ruta in [ruta_es, ruta_la, ruta_scores]):

    try:

        with open(ruta_es, 'r', encoding='utf-8') as f_es, \
             open(ruta_la, 'r', encoding='utf-8') as f_la, \
             open(ruta_scores, 'r', encoding='utf-8') as f_scores:
            
            # .strip() elimina los saltos de línea (\n) al final de cada oración
            textos_es = [linea.strip() for linea in f_es]
            textos_la = [linea.strip() for linea in f_la]
            # Convertimos los scores a números flotantes
            scores = [float(linea.strip()) for linea in f_scores]

        #  importante confirmar que el paralelismo es perfecto
        if not (len(textos_es) == len(textos_la) == len(scores)):
            raise ValueError("Los archivos no tienen la misma cantidad de líneas. El corpus está desalineado.")

        #  Construir el objeto Dataset de Hugging Face
        # Esto agrupa nuestras listas independientes en el formato optimizado que necesitamos
        dataset = Dataset.from_dict({
            "es": textos_es,
            "la": textos_la,
            "score": scores
        })

        print(f"\nDataset construido exitosamente desde los archivos paralelos :D ")
        print(f" Total de pares de oraciones disponibles en memoria: {len(dataset):,}")

        # muestreo 
        print("\n 5 Pares de Oraciones Aleatorias")
        indices_aleatorios = random.sample(range(len(dataset)), 5)

        for i, idx in enumerate(indices_aleatorios):
            fila = dataset[idx]
            print(f"\n Ejemplo {i+1} (Índice {idx}):")
            print(f"    Latín:   {fila['la']}")
            print(f"    Español: {fila['es']}")
            print(f"    Score:   {fila['score']}")

    except Exception as e:
        print(f"\n error {e}")

else:
    print("la ruta esta incorrecta o faltan archivos")

 Leer los archivos línea por línea simultáneamente
Usamos encoding='utf-8' para evitar problemas con tildes o caracteres especiales
la libreria Dataset es de Hugging Face se usa para organizar los datos de manera eficiente y compatible con modelos de NLP ( Natural Language Processing)
en este caso en especifico agrupa las listas de textos y scores en un formato tabular


In [ ]:
import re

print("Iniciando el proceso de limpieza y filtrado del dataset...")

#  hiperparámetros de impieza 
MIN_SCORE = 1.05 

MIN_PALABRAS = 3
MAX_PALABRAS = 50

# Máxima desproporción permitida (ej. 2.0 significa que una oración 
# no puede tener más del doble de palabras que su contraparte)
MAX_PROPORCION = 2.0

def limpiar_par_oraciones(fila):
    # Regla 1: Filtro de puntuación de alineación
    if fila['score'] < MIN_SCORE:
        return False
        
    texto_la = fila['la']
    texto_es = fila['es']
    
    #  filtro de caracteres 
    if "http" in texto_la or "http" in texto_es or "<" in texto_la:
        return False
        
    # filtro de longitud 
    palabras_la = texto_la.split()
    palabras_es = texto_es.split()
    
    len_la = len(palabras_la)
    len_es = len(palabras_es)
    
    if len_la < MIN_PALABRAS or len_la > MAX_PALABRAS:
        return False
    if len_es < MIN_PALABRAS or len_es > MAX_PALABRAS:
        return False
        

    proporcion = max(len_la, len_es) / min(len_la, len_es)
    if proporcion > MAX_PROPORCION:
        return False
        
    return True

# filtro con multiprocesamiento pq 587mil lineas es demasiao papito, pero todavia no es tan pesado como el entrewnamiento, eso si pesa, pa eso esta tu gpu chocotavo

num_nucleos = os.cpu_count() or 2
dataset_limpio = dataset.filter(limpiar_par_oraciones, num_proc=num_nucleos)

print("\n ¡Limpieza completada!")
print(f" Pares originales: {len(dataset):,}")
print(f" Pares limpios retenidos: {len(dataset_limpio):,}")
print(f" Pares descartados: {len(dataset) - len(dataset_limpio):,}")


que significa es min score?
 Umbral de calidad del score (depende de cómo se generó CCMatrix, 
 valores superiores a 1.04 - 1.06 suelen indicar buena calidad en LASER

MAX_PROPORCION 
una regla de un cuya referencia perdi de porque es importante usar esto es "basura entra basura sale" //ewe al que documente, si lo busca para referenciarlo estaria joya, o algun otro texto que lo mencione
sin esto podria entrar una oracion en latin de 1 palabra y su contraparte 30, esto nos daria tremendas alucinadas
 2.0 significa que una oración 
 no puede tener más del doble de palabras que su contraparte)



